# Session 9.3: FastAPI for ML Models

## Learning Objectives
1. Understand FastAPI advantages over Flask
2. Build ML APIs with automatic validation
3. Use Pydantic models for type safety
4. Generate automatic API documentation
5. Deploy image classifier with FastAPI

**Duration:** 2 hours (60 min video + 60 min hands-on)

---

## Why FastAPI?

### FastAPI vs Flask

| Feature | Flask | FastAPI |
|---------|-------|----------|
| Speed | Moderate | Very Fast (async support) |
| Type Hints | No | Yes (enforced) |
| Validation | Manual | Automatic (Pydantic) |
| Documentation | Manual | Automatic (Swagger/OpenAPI) |
| Async | Limited | Full support |
| Learning Curve | Easy | Medium |

**When to use FastAPI:**
- Production ML APIs
- Need automatic validation
- Want interactive documentation
- Performance critical
- Modern Python projects

**When to use Flask:**
- Simple projects
- Quick prototypes
- Team familiar with Flask
- Integration with Flask ecosystem

## Part 1: FastAPI Basics

### Installation

In [ ]:
# Install FastAPI and dependencies
# Run in terminal or uncomment below
# !pip install fastapi uvicorn python-multipart pydantic

### Simple FastAPI Application

**LLM Prompt:**
```
Create a simple FastAPI application with:
- A GET endpoint at root ("/") returning {"message": "Hello World"}
- A GET endpoint at "/health" returning {"status": "ok"}
- Include proper type hints
```

In [ ]:
# Save this as: fastapi_basic_app.py

from fastapi import FastAPI
from pydantic import BaseModel
from typing import Dict

# Create FastAPI instance
app = FastAPI(
    title="ML API",
    description="Machine Learning API with FastAPI",
    version="1.0.0"
)

@app.get("/")
async def root() -> Dict[str, str]:
    """Root endpoint"""
    return {"message": "Hello World", "docs": "/docs"}

@app.get("/health")
async def health() -> Dict[str, str]:
    """Health check endpoint"""
    return {"status": "ok"}

# Run with: uvicorn fastapi_basic_app:app --reload
# Access docs at: http://localhost:8000/docs

## Part 2: Pydantic Models for Validation

### Define Input/Output Models

In [ ]:
from pydantic import BaseModel, Field, validator
from typing import List, Optional

# Input model for image classification
class ImageInput(BaseModel):
    """Input schema for image classification"""
    image_data: str = Field(..., description="Base64 encoded image")
    model_version: Optional[str] = Field("v1", description="Model version to use")
    
    @validator('image_data')
    def validate_image(cls, v):
        if len(v) < 100:
            raise ValueError('Image data too small')
        return v

# Output model for predictions
class PredictionOutput(BaseModel):
    """Output schema for predictions"""
    class_name: str = Field(..., description="Predicted class")
    confidence: float = Field(..., ge=0.0, le=1.0, description="Prediction confidence")
    top_5_classes: List[Dict[str, float]] = Field(..., description="Top 5 predictions")
    model_version: str = Field(..., description="Model version used")

# Example usage
example_input = {
    "image_data": "base64encodedstring" * 10,
    "model_version": "v1"
}

# Pydantic validates automatically
validated = ImageInput(**example_input)
print(validated.dict())

## Part 3: Image Classifier API with FastAPI

### Complete ML API Implementation

**LLM Prompt:**
```
Create a FastAPI application for image classification with:
- POST /predict endpoint accepting image file upload
- Use pre-trained MobileNetV2 from Keras
- Return top 5 predictions with probabilities
- Input validation using Pydantic
- Error handling with proper HTTP status codes
- Automatic Swagger documentation
```

In [ ]:
# Save this as: image_classifier_api/main.py

from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from typing import List, Dict
import numpy as np
from PIL import Image
import io

# For MobileNetV2
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input, decode_predictions

# Create FastAPI app
app = FastAPI(
    title="Image Classification API",
    description="ML API for image classification using MobileNetV2",
    version="1.0.0"
)

# Enable CORS
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Load model at startup
model = None

@app.on_event("startup")
async def load_model():
    """Load model when API starts"""
    global model
    model = MobileNetV2(weights='imagenet')
    print("Model loaded successfully!")

# Response models
class Prediction(BaseModel):
    class_name: str
    class_id: str
    confidence: float = Field(..., ge=0.0, le=1.0)

class PredictionResponse(BaseModel):
    success: bool
    predictions: List[Prediction]
    model_version: str = "MobileNetV2"

class ErrorResponse(BaseModel):
    success: bool = False
    error: str

# Helper function
def preprocess_image(image_bytes: bytes) -> np.ndarray:
    """Preprocess image for MobileNetV2"""
    try:
        # Open image
        image = Image.open(io.BytesIO(image_bytes))
        
        # Convert to RGB if necessary
        if image.mode != 'RGB':
            image = image.convert('RGB')
        
        # Resize to 224x224
        image = image.resize((224, 224))
        
        # Convert to array
        img_array = np.array(image)
        
        # Add batch dimension
        img_array = np.expand_dims(img_array, axis=0)
        
        # Preprocess for MobileNetV2
        img_array = preprocess_input(img_array)
        
        return img_array
    except Exception as e:
        raise ValueError(f"Error preprocessing image: {str(e)}")

@app.get("/", response_model=Dict[str, str])
async def root():
    """Root endpoint with API info"""
    return {
        "message": "Image Classification API",
        "docs": "/docs",
        "predict": "/predict"
    }

@app.get("/health")
async def health():
    """Health check endpoint"""
    return {
        "status": "ok",
        "model_loaded": model is not None
    }

@app.post("/predict", response_model=PredictionResponse, responses={400: {"model": ErrorResponse}})
async def predict(file: UploadFile = File(...)):
    """
    Classify an uploaded image
    
    - **file**: Image file (JPG, PNG)
    
    Returns top 5 predictions with confidence scores
    """
    # Validate file type
    if file.content_type not in ["image/jpeg", "image/png", "image/jpg"]:
        raise HTTPException(
            status_code=400,
            detail="Invalid file type. Please upload JPG or PNG image."
        )
    
    try:
        # Read image bytes
        image_bytes = await file.read()
        
        # Preprocess image
        processed_image = preprocess_image(image_bytes)
        
        # Make prediction
        predictions = model.predict(processed_image)
        
        # Decode predictions
        decoded = decode_predictions(predictions, top=5)[0]
        
        # Format response
        results = [
            Prediction(
                class_id=pred[0],
                class_name=pred[1],
                confidence=float(pred[2])
            )
            for pred in decoded
        ]
        
        return PredictionResponse(
            success=True,
            predictions=results
        )
        
    except ValueError as e:
        raise HTTPException(status_code=400, detail=str(e))
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Prediction error: {str(e)}")

# Run with: uvicorn main:app --reload
# Docs at: http://localhost:8000/docs

## Part 4: Testing the FastAPI Application

### Test with Python Requests

In [ ]:
import requests
from PIL import Image
import io

# Test health endpoint
response = requests.get("http://localhost:8000/health")
print("Health:", response.json())

# Test prediction endpoint
# Download a test image or use local file
image_path = "test_image.jpg"  # Replace with actual image

with open(image_path, "rb") as f:
    files = {"file": ("image.jpg", f, "image/jpeg")}
    response = requests.post("http://localhost:8000/predict", files=files)

if response.status_code == 200:
    result = response.json()
    print("\nTop 5 Predictions:")
    for pred in result["predictions"]:
        print(f"  {pred['class_name']}: {pred['confidence']:.2%}")
else:
    print("Error:", response.json())

## Part 5: Automatic Documentation

### Interactive API Documentation

FastAPI automatically generates:

1. **Swagger UI** at `/docs`
   - Interactive API testing
   - Try endpoints directly
   - See request/response schemas

2. **ReDoc** at `/redoc`
   - Alternative documentation view
   - Better for reading
   - Clean layout

3. **OpenAPI Schema** at `/openapi.json`
   - Machine-readable API spec
   - For code generation
   - Integration with tools

**Try it:**
1. Run the API: `uvicorn main:app --reload`
2. Open browser: http://localhost:8000/docs
3. Click on POST /predict
4. Click "Try it out"
5. Upload an image
6. Click "Execute"
7. See results!

## Part 6: Async Support for Performance

### Why Async Matters for ML APIs

In [ ]:
# Regular synchronous endpoint (blocking)
@app.post("/predict-sync")
def predict_sync(file: UploadFile = File(...)):
    # This blocks the entire server while processing
    result = model.predict(processed_image)
    return result

# Async endpoint (non-blocking)
@app.post("/predict-async")
async def predict_async(file: UploadFile = File(...)):
    # Can handle other requests while waiting
    image_bytes = await file.read()  # Non-blocking I/O
    result = model.predict(processed_image)  # Still blocking (CPU-bound)
    return result

# For CPU-bound ML inference, use background tasks
from fastapi import BackgroundTasks

@app.post("/predict-background")
async def predict_background(file: UploadFile, background_tasks: BackgroundTasks):
    # Process in background
    background_tasks.add_task(process_and_log, file)
    return {"message": "Processing started"}

def process_and_log(file):
    # Heavy processing here
    pass

## Part 7: Error Handling and Validation

### Custom Exception Handlers

In [ ]:
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import JSONResponse
from fastapi.exceptions import RequestValidationError

# Custom exception
class ModelNotLoadedError(Exception):
    def __init__(self, message: str):
        self.message = message

# Exception handler
@app.exception_handler(ModelNotLoadedError)
async def model_not_loaded_handler(request: Request, exc: ModelNotLoadedError):
    return JSONResponse(
        status_code=503,
        content={"error": exc.message, "detail": "Model initialization failed"}
    )

# Validation error handler
@app.exception_handler(RequestValidationError)
async def validation_exception_handler(request: Request, exc: RequestValidationError):
    return JSONResponse(
        status_code=422,
        content={
            "error": "Validation Error",
            "details": exc.errors()
        }
    )

## Hands-On Exercise: Build Your Own FastAPI ML Service

### Task: Create a Sentiment Analysis API

**Requirements:**
1. POST /analyze endpoint
2. Input: text (string)
3. Output: sentiment (positive/negative), confidence (0-1)
4. Use your trained sentiment model from Module 8
5. Pydantic validation
6. Error handling
7. Automatic docs

**LLM Prompt:**
```
Create a FastAPI application for sentiment analysis:
- POST /analyze endpoint accepting text input
- Use Pydantic model for input validation (text length 1-500 chars)
- Load pre-trained sentiment model
- Return sentiment (positive/negative) and confidence score
- Include error handling for empty text and model errors
- Add CORS middleware
- Generate automatic Swagger documentation
```

**Time:** 60 minutes

In [ ]:
# Your code here
# Save as: sentiment_api/main.py

# TODO: Implement sentiment analysis API

## Part 8: Deployment Preparation

### Create Requirements File

In [ ]:
# Create requirements.txt
requirements = """
fastapi==0.104.1
uvicorn[standard]==0.24.0
python-multipart==0.0.6
pydantic==2.5.0
tensorflow==2.14.0
pillow==10.1.0
numpy==1.24.3
"""

with open("requirements.txt", "w") as f:
    f.write(requirements.strip())

print("requirements.txt created!")

### Create README

In [ ]:
readme = """
# Image Classification API

FastAPI-based ML API for image classification using MobileNetV2.

## Features
- Fast and async
- Automatic validation
- Interactive API docs
- Pre-trained MobileNetV2
- Top 5 predictions

## Setup

```bash
pip install -r requirements.txt
```

## Run

```bash
uvicorn main:app --reload
```

## API Documentation

- Swagger UI: http://localhost:8000/docs
- ReDoc: http://localhost:8000/redoc

## Usage

```python
import requests

with open("image.jpg", "rb") as f:
    response = requests.post(
        "http://localhost:8000/predict",
        files={"file": f}
    )
print(response.json())
```
"""

with open("README.md", "w") as f:
    f.write(readme.strip())

print("README.md created!")

## Summary

### What You Learned
✅ FastAPI advantages over Flask
✅ Pydantic models for validation
✅ Building ML APIs with FastAPI
✅ Automatic documentation generation
✅ Async support for performance
✅ Error handling and custom exceptions
✅ Deployment preparation

### Key Takeaways
1. **FastAPI is ideal for production ML APIs**
2. **Automatic validation saves development time**
3. **Interactive docs improve API usability**
4. **Type hints ensure code quality**
5. **Async support improves performance**

### Next Session
**Session 9.4:** Cloud Deployment (Heroku)
- Deploy FastAPI to Heroku
- Public API access
- Production configuration

---

## Troubleshooting

**Issue:** Model too large for memory
- **Solution:** Use model quantization or lighter model

**Issue:** Slow prediction times
- **Solution:** Use background tasks or async workers

**Issue:** File upload fails
- **Solution:** Check file size limit in FastAPI config

**Issue:** CORS errors
- **Solution:** Add CORSMiddleware with proper origins

---

**Session Duration:** ~120 minutes
**Difficulty:** Medium
**Prerequisites:** Python, basic ML, Session 9.2 (Flask)
"